In [1]:
# libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

In [2]:
file_path = "../data/raw/Guns incident Data.csv"
df = pd.read_csv(file_path)

In [ ]:
missing_summary = df.isna().sum().to_frame('Missing_Count')
missing_summary['Missing_%'] = (missing_summary['Missing_Count'] / len(df)) * 100
missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values(by='Missing_%', ascending=False)
missing_summary


In [ ]:
# Convert missing to boolean (1 if missing)
missing_corr = df.isna().corr()

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
sns.heatmap(missing_corr, cmap="coolwarm", center=0)
plt.title("Correlation Between Missing Values")
plt.show()


In [ ]:
df['missing_count'] = df.isna().sum(axis=1)
df['missing_count'].value_counts().sort_index()


In [ ]:
# ===============================================
# 📘 Guns Dataset - Missing Value Analysis & Handling
# ===============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load dataset ---
df = pd.read_csv("guns.csv")

# ===============================================
# Step 1: Overview of Missing Values
# ===============================================
missing_summary = (
    df.isnull().sum()
    .to_frame('Missing_Count')
    .assign(Missing_%=lambda x: 100 * x['Missing_Count'] / len(df),
            Dtype=df.dtypes)
    .query("Missing_Count > 0")
)
print("📊 Missing Value Summary:")
display(missing_summary)

# --- Optional: Pretty markdown-style output ---
from IPython.display import Markdown
md_table = missing_summary.reset_index().rename(columns={'index':'Column'}).to_markdown(index=False)
display(Markdown(md_table))


# ===============================================
# Step 2: Trend of Missing Values over Time
# ===============================================
df['Year'] = df['Year'].astype(int)

# Missing rate by year for each affected column
missing_by_year = df.groupby('Year')[['Education', 'Place of incident', 'Age']].apply(
    lambda x: x.isna().mean() * 100
)

missing_by_year.plot(marker='o', figsize=(10, 6))
plt.title("Missing Value Trend (%) by Year")
plt.ylabel("Missing (%)")
plt.xlabel("Year")
plt.legend(title="Column")
plt.show()


# ===============================================
# Step 3: Missingness by Race
# ===============================================

# Create missing flags
df['Education_missing'] = df['Education'].isna()
df['Place_missing'] = df['Place of incident'].isna()

edu_missing_by_race = (
    df.groupby('Race')['Education_missing']
    .mean().mul(100)
    .sort_values(ascending=False)
)
place_missing_by_race = (
    df.groupby('Race')['Place_missing']
    .mean().mul(100)
    .sort_values(ascending=False)
)

print("Education Missing % by Race:")
display(edu_missing_by_race)

print("\nPlace of Incident Missing % by Race:")
display(place_missing_by_race)

# Markdown-friendly table
edu_md = edu_missing_by_race.reset_index().rename(columns={'Education_missing':'Education_missing (%)'}).to_markdown(index=False)
place_md = place_missing_by_race.reset_index().rename(columns={'Place_missing':'Place_missing (%)'}).to_markdown(index=False)
display(Markdown("**Education Missing % by Race**\n" + edu_md))
display(Markdown("**Place of Incident Missing % by Race**\n" + place_md))


# ===============================================
# Step 4: Correlation Between Missingness
# ===============================================
df['Age_missing'] = df['Age'].isna().astype(int)
df['Education_missing'] = df['Education'].isna().astype(int)
df['Place_missing'] = df['Place of incident'].isna().astype(int)

corr_missing = df[['Age_missing', 'Education_missing', 'Place_missing']].corr()
print("\n🔗 Correlation between Missingness:")
display(corr_missing)

# ===============================================
# Step 5: Handling Missing Values
# ===============================================

# Check skewness of Age
skewness = df['Age'].skew()
print(f"\nSkewness of Age: {skewness:.4f}")

# Age: use mean since skewness < 0.5
df['Age'] = df['Age'].fillna(df['Age'].mean())

# Education: use a placeholder for unknown values
df['Education'] = df['Education'].fillna('Unknown')

# Place of incident: use a placeholder for unspecified
df['Place of incident'] = df['Place of incident'].fillna('Unspecified')

# Verify no missing values remain
print("\n✅ Remaining Missing Values:")
display(df.isna().sum())

print("\n🎯 Data Cleaning Completed Successfully.")
